# RAGAS EVALUATION

## Imports, Functions, Settings

In [ ]:
	
import asyncio
import sys
import os
from pathlib import Path
import dotenv
import logging
from typing import List, Dict, Optional, Any
import json

# Ensure we're in the right directory and add to path
notebook_dir = Path.cwd()
project_root = notebook_dir.parent  # Go up to project root
os.chdir(str(project_root))
sys.path.insert(0, str(project_root))

print(f"Working directory: {os.getcwd()}")
print(f"Python path includes: {project_root}")

from agent.graph_utils import initialize_graph, test_graph_connection, close_graph
from agent.vector_db_utils import initialize_database, close_database

from agent.single_agent import single_agent_run
from agent.multi_agent_router.orchestrator import multi_agent_router_orchestrator_run


print("All imports successful")

In [ ]:
from __future__ import annotations

from pathlib import Path
import pandas as pd

def write_csv(obj, filename: str, out_dir: str | Path | None = None, *, index: bool = False) -> Path:
    """Write `obj` as a CSV into evaluation/ragas/ (relative to project_root if available).
    
    `obj` can be a pandas DataFrame, or something coercible into one (list[dict], dict, etc.).
    Returns the written file path.
    """
    if isinstance(obj, pd.DataFrame):
        df_to_write = obj
    else:
        df_to_write = pd.DataFrame(obj)

    base_dir = None
    if out_dir is not None:
        base_dir = Path(out_dir)
    elif "project_root" in globals():
        base_dir = Path(project_root) / "evaluation" / "ragas"
    else:
        base_dir = Path.cwd() / "evaluation" / "ragas"

    base_dir.mkdir(parents=True, exist_ok=True)
    if not filename.lower().endswith(".csv"):
        filename = f"{filename}.csv"

    path = base_dir / filename
    df_to_write.to_csv(path, index=index)
    print(f"Wrote CSV: {path} ({len(df_to_write)} rows)")
    return path

In [ ]:
# -----------------------------
# Libaries & Settings
# -----------------------------
import os
import logging
from dataclasses import is_dataclass, fields as dataclass_fields
from typing import List, Dict, Optional, Any, Tuple
from pydantic import BaseModel
from ragas.embeddings import embedding_factory
from ragas.llms import llm_factory
from ragas import experiment, Dataset
from  ragas.metrics.collections import Faithfulness, AnswerRelevancy, ContextRelevance
from dotenv import load_dotenv
# from ragas.metrics import AnswerRelevancy
# from ragas.embeddings import embedding_factory
from openai import AsyncOpenAI

from ragas.embeddings import OpenAIEmbeddings

load_dotenv()


api_key = os.getenv("OPENAI_API_KEY")
client_openai = AsyncOpenAI(api_key=os.getenv("OPENAI_API_KEY"))

embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small",
    client=client_openai,
 )
llm = llm_factory("gpt-4o-mini" , client=client_openai, max_tokens=8000)

# -----------------------------
# Data Set Prep
# -----------------------------
def prepare_dataset_ragas( 
        kpi_names: list[str], 
        years: list[str],
        reranker_entity_list: list[str], 
        reranker_fact_list: list[str],
        limit_vector_results_list: list[int],
        alpha_hybrid_list: list[float],
        limit_fact_list: list[int],
        limit_entity_list: list[int],
        k_list: list[int],
        queries: List[str],
        generated_answers: Optional[List[str]],
        retrieved_contexts: List[List[str]],
        kpi_names_expected: Optional[List[str]] = None,
        kpi_names_agent: Optional[List[str]] = None,
        name: str = "evaluation",
        client: Optional[str] = None, 
        agent: Optional[str] = None
    ) -> List[Dict[str, Any]]:  
        if client == "default_client": client = "RWE"
        dataset = Dataset(
            name=name,
            backend="local/csv",
            root_dir="./data/ragas/"
        )
        if generated_answers is not None:
            if kpi_names_expected is None:
                kpi_names_expected = [""] * len(queries)
            if kpi_names_agent is None:
                kpi_names_agent = [""] * len(queries)
            i = 1
            for q, a, ctxs, kpi_name, kpi_name_expected, kpi_name_agent, year, reranker_entity, reranker_fact, limit_vector_results, alpha_hybrid, limit_fact, limit_entity, k in zip(
                queries, generated_answers, retrieved_contexts, kpi_names, kpi_names_expected, kpi_names_agent, years, 
                reranker_entity_list, reranker_fact_list,
                limit_vector_results_list, alpha_hybrid_list,
                limit_fact_list, limit_entity_list, k_list):
                dataset.append(
                    {   "index": i,
                        "kpi_name": kpi_name,
                        "kpi_name_expected": kpi_name_expected,
                        "kpi_name_agent": kpi_name_agent,
                        "fiscal_year": year,

                        "reranker_entity_list": reranker_entity,
                        "reranker_fact_list": reranker_fact,
                        "limit_vector_results_list": limit_vector_results,
                        "alpha_hybrid_list": alpha_hybrid,
                        "limit_fact_list": limit_fact,
                        "limit_entity_list": limit_entity,
                        "k_list": k,
                        
                        "user_input": q,
                        "response": a,
                        "retrieved_contexts": ctxs,
                    }
                )
                i=i+1
        dataset.save()
        return dataset

# -----------------------------
# Agent Run and Data Set Prep
# -----------------------------

def _ensure_string(value: Any) -> str:
    """Convert value to string. If it's a list, join with newlines."""
    if value is None:
        return ""
    if isinstance(value, str):
        return value
    if isinstance(value, list):
        return "\n".join(str(item) for item in value)
    return str(value)


def _ensure_context_list(value: Any) -> List[str]:
    """Coerce retrieved contexts to the RAGAS-expected shape: list[str]."""
    if value is None:
        return []
    if isinstance(value, str):
        return [value]
    if isinstance(value, list):
        flattened: List[str] = []
        for item in value:
            if item is None:
                continue
            if isinstance(item, str):
                flattened.append(item)
                continue
            if isinstance(item, list):
                for sub in item:
                    if sub is None:
                        continue
                    flattened.append(sub if isinstance(sub, str) else str(sub))
                continue
            flattened.append(str(item))
        return flattened
    try:
        return [str(v) for v in value]  # type: ignore[arg-type]
    except Exception:
        return [str(value)]


def kpi_queries_and_expected_names(
    kpi_obj: Any,
    *,
    strip_year_suffix: bool = True,
    ) -> Tuple[List[str], List[str]]:
    """
    Return (queries, expected_kpi_names) aligned by dataclass field order.

    For the quantitative KPI dataclasses, fields are often named like `EBITDA_0/1/2`.
    With `strip_year_suffix=True` this becomes `EBITDA` for all three queries.
    """
    if not is_dataclass(kpi_obj):
        raise TypeError(f"Expected a dataclass instance, got: {type(kpi_obj)!r}")

    queries: List[str] = []
    expected_names: List[str] = []
    for field in dataclass_fields(kpi_obj):
        if field.name in ("client", "_built"):
            continue
        value = getattr(kpi_obj, field.name)
        if value is None:
            continue
        name = field.name
        if strip_year_suffix and len(name) > 2 and name[-2] == "_" and name[-1].isdigit():
            name = name[:-2]
        expected_names.append(name)
        queries.append(str(value))
    return queries, expected_names


async def agent_run_eval(
    query_list: List[str],
    agent: str = "single",
    type: str = "quantitative",
    name: str = "evaluation",
    client: str = "default_client",
    override_kpi_names: Optional[List[str]] = None,
    reranker_entity: str = "RRF",
    reranker_fact: str = "RRF",
    limit_vector_results: int = 15,
    alpha_hybrid: float = 0.6,
    limit_fact: int = 30,
    limit_entity: int = 10,
    k: int = 2,
 ) -> List[Dict[str, Any]]:
    logger = logging.getLogger(__name__)
    log_level = logging.INFO
    logging.basicConfig(
        level=log_level,
        format="%(asctime)s - %(name)s - %(levelname)s - %(message)s",
    )
    logging.getLogger("httpx").setLevel(logging.WARNING)

    if override_kpi_names is not None and len(override_kpi_names) != len(query_list):
        raise ValueError(
            f"Length mismatch: override_kpi_names has {len(override_kpi_names)} entries, "
            f"but query_list has {len(query_list)} queries."
        )
    try:
        await initialize_graph()
        logger.info("Graph database initialized")

        graph_ok = await test_graph_connection()
        if not graph_ok:
            logger.error("Graph database connection test failed.")
    except Exception as e:
        logger.error(f"Failed to initialize graph database: {e}")
        raise
    try:
        initialize_database()
        logger.info("Vector database initialized")
    except Exception as e:
        logger.error(f"Failed to initialize vector database: {e}")
        raise

    # Create agent instance and run
    answers: List[Any] = []
    contexts_list: List[Any] = []
    kpi_names: List[Any] = []
    kpi_names_expected_list: List[Any] = []
    kpi_names_agent_list: List[Any] = []
    years: List[Any] = []
    reranker_entity_list: List[str] = []
    reranker_fact_list: List[str] = []
    limit_vector_results_list: List[int] = []
    alpha_hybrid_list: List[float] = []
    limit_fact_list: List[int] = []
    limit_entity_list: List[int] = []
    k_list: List[int] = []
    type_retrieval_list: List[Any] = []

    try:
        if agent == "single_agent":
            eval_agent = single_agent_run(
                client=client,
                reranker_entity=reranker_entity,
                reranker_fact=reranker_fact,
                limit_vector_results=limit_vector_results,
                alpha_hybrid=alpha_hybrid,
                limit_fact=limit_fact,
                limit_entity=limit_entity,
                k=k,
            )
        elif agent == "multi_agent":
            eval_agent = multi_agent_router_orchestrator_run(
                client=client,
                reranker_fact=reranker_fact,
                reranker_entity=reranker_entity,
                limit_vector_results=limit_vector_results,
                alpha_hybrid=alpha_hybrid,
                limit_fact=limit_fact,
                limit_entity=limit_entity,
                k=k,
            )
        else:
            raise ValueError(f"Unsupported agent type: {agent}")

        for idx, query in enumerate(query_list):
            
            expected_kpi_name = _ensure_string(override_kpi_names[idx]) if override_kpi_names is not None else ""

            result = await eval_agent.run_eval([query],expected_kpi_name )

            agent_kpi_name = _ensure_string(result.get("kpi_name"))
            kpi_names_agent_list.append(agent_kpi_name)

            kpi_names_expected_list.append(expected_kpi_name)
            # Main KPI name used for grouping: prefer expected, else fallback to agent
            kpi_names.append(expected_kpi_name or agent_kpi_name)

            years.append(result.get("year"))
            reranker_entity_list.append(reranker_entity)
            reranker_fact_list.append(reranker_fact)
            limit_vector_results_list.append(limit_vector_results)
            alpha_hybrid_list.append(alpha_hybrid)
            limit_fact_list.append(limit_fact)
            limit_entity_list.append(limit_entity)
            k_list.append(k)
            
            # Get current result's type_retrieval (not the list!)
            current_type_retrieval = result.get("type_retrieval")
            type_retrieval_list.append(current_type_retrieval)

            if agent == "single_agent":
                answers.append(_ensure_string(result["answer"]))
                contexts_list.append(_ensure_context_list(result.get("contexts")))
            elif agent == "multi_agent":
                # Compare with current_type_retrieval, not the list
                # Use _ensure_string to convert list responses to string for RAGAS
                if current_type_retrieval == "quantitative":
                    answers.append(_ensure_string(result["quantitative_findings"]))
                    contexts_list.append(_ensure_context_list(result.get("quantitative_context")))
                elif current_type_retrieval == "qualitative":
                    answers.append(_ensure_string(result["qualitative_findings"]))
                    contexts_list.append(_ensure_context_list(result.get("qualitative_context")))
                elif current_type_retrieval == "writer":
                    answers.append(_ensure_string(result["report"]))
                    contexts_list.append(_ensure_context_list(result.get("all_contexts")))
        print("Agent Response Generated")
    except Exception as e:
        print(f"Error running agent: {e}")
        import traceback
        traceback.print_exc()
        result = None
    await close_graph()
    close_database()
    logger.info("Graph and vector database connections closed")

    dataset = prepare_dataset_ragas(
        queries=query_list,
        kpi_names=kpi_names,
        kpi_names_expected=kpi_names_expected_list,
        kpi_names_agent=kpi_names_agent_list,
        years=years,
        generated_answers=answers,
        retrieved_contexts=contexts_list,
        name=name,
        reranker_entity_list=reranker_entity_list,
        reranker_fact_list=reranker_fact_list,
        limit_vector_results_list=limit_vector_results_list,
        alpha_hybrid_list=alpha_hybrid_list,
        limit_fact_list=limit_fact_list,
        limit_entity_list=limit_entity_list,
        k_list=k_list,
    )
    return dataset

# -----------------------------
# Experiment (truncate inputs to avoid max_tokens failures)
# -----------------------------

def _clip_text(value: Any, max_chars: int) -> str:
    if value is None:
        return ""
    text = str(value)
    if len(text) <= max_chars:
        return text
    return text[:max_chars] + "...[TRUNCATED]"


def _clip_contexts(
    contexts: Any,
    *,
    max_contexts: int = 30,
    max_chars_each: int = 800,
    max_total_chars: int = 10000000,
 ) -> List[str]:
    if contexts is None:
        return []
    if isinstance(contexts, str):
        contexts_list = [contexts]
    else:
        try:
            contexts_list = list(contexts)
        except Exception:
            contexts_list = [str(contexts)]

    clipped: List[str] = []
    total = 0
    for ctx in contexts_list[:max_contexts]:
        clipped_ctx = _clip_text(ctx, max_chars_each)
        if not clipped_ctx:
            continue
        if total + len(clipped_ctx) > max_total_chars:
            break
        clipped.append(clipped_ctx)
        total += len(clipped_ctx)
    return clipped


@experiment()
async def run_evaluation(row: Dict[str, Any], name: str = "baseline") -> Dict[str, Any]:
    """
    Evaluate RAGAS metrics for a single row.

    NOTE: We truncate long `response` / `retrieved_contexts` to prevent the
    underlying LLM graders (e.g., Faithfulness statement extraction) from
    exceeding model `max_tokens` and failing.
    """

    print("Evaluating row:", row)
    row = dict(row)
    #row["user_input"] = _clip_text(row.get("user_input", ""), 300)
    #row["response"] = _clip_text(row.get("response", ""), 1400)
    #row["retrieved_contexts"] = _clip_contexts(row.get("retrieved_contexts", []))
    #max_tokens=2048
    faithfulness = Faithfulness(llm=llm)
    context_relevance = ContextRelevance(llm=llm)
    answer_relevancy = AnswerRelevancy(llm=llm, embeddings=embeddings)

    faith_result = await faithfulness.ascore(
        user_input=row["user_input"],
        response=row["response"],
        retrieved_contexts=row["retrieved_contexts"],
    )

    context_result = await context_relevance.ascore(
        user_input=row["user_input"],
        retrieved_contexts=row["retrieved_contexts"],
    )

    answer_result = await answer_relevancy.ascore(
        user_input=row["user_input"],
        response=row["response"],
    )
    return_list = {
        "index": row.get("index", -1),
        "kpi_name": row.get("kpi_name", ""),
        "faithfulness": float(faith_result.value),
        "context_relevance": float(context_result.value),
        "answer_relevancy": float(answer_result.value),
        "experiment_name": name,
        **row,
    }

    return return_list  
        

## QUERY SETS


In [ ]:
# PLEASE insert your client company here: "RWE" or "Walmart"
client_company = ""

In [ ]:
# -----------------------------
# Query Sets
# -----------------------------

# Make the cell robust if the kernel was restarted and the path-init cell wasn't re-run.
import os
import sys
from pathlib import Path


client = client_company

# PLEASE UPDATE THIS FUNCTION IF YOUR PROJECT ROOT IS NOT BEING DETECTED CORRECTLY.
def _find_project_root() -> Path:
    def _has_markers(candidate: Path) -> bool:
        return (candidate / "agent").is_dir() and (candidate / "requirements.txt").is_file()

    # 1) Fast path: cwd + parents
    cwd = Path.cwd().resolve()
    for candidate in (cwd,) + tuple(cwd.parents):
        if _has_markers(candidate):
            return candidate

    # 2) VS Code / shell hints
    env_cwd = os.getenv("VSCODE_CWD") or os.getenv("PWD")
    if env_cwd:
        p = Path(env_cwd).resolve()
        for candidate in (p,) + tuple(p.parents):
            if _has_markers(candidate):
                return candidate

    # 3) macOS iCloud Drive hint (your project lives here)
    icloud_root = Path.home() / "Library" / "Mobile Documents" / "com~apple~CloudDocs"
    if icloud_root.is_dir():
        patterns = [
            "agentic_rag_ingestion",
            "*/agentic_rag_ingestion",
            "*/*/agentic_rag_ingestion",
            "*/*/*/agentic_rag_ingestion",
            "*/*/*/*/agentic_rag_ingestion",
            "*/*/*/*/*/agentic_rag_ingestion",
        ]
        for pat in patterns:
            for candidate in icloud_root.glob(pat):
                if _has_markers(candidate):
                    return candidate

    raise RuntimeError(
        "Could not find project root containing an 'agent/' folder and 'requirements.txt'. "
        "Run Cell 1 (path setup) or update _find_project_root() hints."
    )

project_root = _find_project_root()
os.chdir(project_root)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
print(f"Project root: {project_root}")

from agent.query_qualtitative_kpi import QualitativeKPIs_speedboat


# Qualitative (expected KPI names = dataclass field names)
kpi_qualitative_speedboat = QualitativeKPIs_speedboat(client=client_company)
query_qualitative_speedboat = kpi_qualitative_speedboat.all_queries()
query_qualitative_speedboat_list, kpi_names_expected_qualitative_speedboat = kpi_queries_and_expected_names(
    kpi_qualitative_speedboat, strip_year_suffix=False
 )


# Qualitative KPI RAGAS

In [ ]:
idx = 0

### Single Agent

In [ ]:
dataset_single_qualitative = await agent_run_eval(
    query_list=query_qualitative_speedboat_list,
    override_kpi_names=kpi_names_expected_qualitative_speedboat,
    agent="single_agent",
    type="qualitative",
    name=f"{client_company}_single_agent_qualitative_speedboat_{idx}",
    client=client,
    alpha_hybrid=0.6,)

In [ ]:
result_single_qualitative = await run_evaluation.arun(dataset_single_qualitative, name=f"{client_company}_single_agent_qualitative_speedboat_{idx}")
result_single_qualitative = sorted(result_single_qualitative, key=lambda r: int(r.get("index", 0)))
write_csv(result_single_qualitative, f"single_agent_{client_company}_qualitative_speedboat_results{idx}.csv")

In [ ]:
for i in result_single_qualitative:
    print(f"Index: {i['index']}")
    print(f"KPI Name: {i['kpi_name']}")
    print(f"Faithfulness: {i['faithfulness']}")
    print(f"Context Relevance: {i['context_relevance']}")
    print(f"Answer Relevancy: {i['answer_relevancy']}")
    print("--------------------------------------------------")

### Multi Agent

In [ ]:
dataset_multi_qualitative  = await agent_run_eval(
    query_list=query_qualitative_speedboat_list,
    override_kpi_names=kpi_names_expected_qualitative_speedboat,
    agent="multi_agent",
    type="qualitative",
    name=f"multi_agent{client_company}_speedboat",
    alpha_hybrid=0.6,
    client=client
    )

In [ ]:
result_mulit_qualitative = await run_evaluation.arun(dataset_multi_qualitative, name=f"multi_agent{client_company}_speedboat")
result_mulit_qualitative = sorted(result_mulit_qualitative, key=lambda r: int(r.get("index", 0)))
write_csv(result_mulit_qualitative, f"multi_agent_{client_company}_qualitative_speedboat_results{idx}.csv")

In [ ]:
print("kpi name ------------- faithfulness, context_relevance, answer_relevancy")
for idx, i in enumerate(result_mulit_qualitative):
    print(idx, "---  ", i["kpi_name"], i["faithfulness"], i["context_relevance"], i["answer_relevancy"])


# PLOTS

In [ ]:
import pandas as pd

def load_experiment_data(experiment_name: str, file_path: Optional[str] = None) -> pd.DataFrame:
    """
    Load experiment data from CSV file into a DataFrame.
    """
    # Use path relative to project root (since cell 1 changes cwd to project root)
    if file_path is None:
        file_path = f"./data/experiments/{experiment_name}.csv"
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"Experiment data file not found: {file_path}")
    df = pd.read_csv(file_path)
    return df



from datetime import datetime
import os
import matplotlib.pyplot as plt


def plot_save(name: str, folder: str = "plots_eval") -> None:
    if not name:
        raise ValueError("Plot name must be provided.")
    os.makedirs(folder, exist_ok=True)
    stamp = datetime.now().strftime("%Y%m%d_%H%M")
    plt.savefig(os.path.join(folder, f"{name}_{stamp}.png"), dpi=300, bbox_inches="tight")

In [ ]:

# PLEASE CHOOSE:
client_name_plot = ""

type_agent = "single_agent"

type_kpi = "qualitative"

metric = "answer_relevance"

In [ ]:
# Change file paths as needed
df_single_agent_qualitative = load_experiment_data(f"{client_name_plot}_single_agent_qualitative", f"./evaluation/ragas/single_agent_{client_name_plot}_speedboat_results.csv")
df_multi_agent_qualitative = load_experiment_data(f"{client_name_plot}_multi_agent_qualitative", f"./evaluation/ragas/multi_agent_{client_name_plot}_qualitative_speedboat_results.csv")

# Rename answer_relevancy to answer_relevance for consistency
df_single_agent_qualitative = df_single_agent_qualitative.rename(columns={'answer_relevancy': 'answer_relevance'})
df_multi_agent_qualitative = df_multi_agent_qualitative.rename(columns={'answer_relevancy': 'answer_relevance'})

# Replace underscores with spaces in kpi_name column
df_single_agent_qualitative['kpi_name'] = df_single_agent_qualitative['kpi_name'].str.replace('_', ' ')
df_multi_agent_qualitative['kpi_name'] = df_multi_agent_qualitative['kpi_name'].str.replace('_', ' ')


In [ ]:
import matplotlib.pyplot as plt

# Get the Blues colormap
blues = plt.cm.Blues

# Get a specific color (value between 0.0 and 1.0)
# 0.0 = lightest, 1.0 = darkest
light_blue = blues(0.3)   # lighter shade
medium_blue = blues(0.5)  # medium shade
dark_blue = blues(0.9)    # darker shade

size_axes = 13
size_subtitle = 14
size_title = 16

## RAGAS PLOTS

In [ ]:
# Merge all metrics
metrics_to_compare = ["faithfulness", "context_relevance", "answer_relevance"]

df_diff = df_single_agent_qualitative[["kpi_name"] + metrics_to_compare].copy()
df_diff = df_diff.rename(columns={m: f"{m}_single" for m in metrics_to_compare})

df_multi_subset = df_multi_agent_qualitative[["kpi_name"] + metrics_to_compare].copy()
df_multi_subset = df_multi_subset.rename(columns={m: f"{m}_multi" for m in metrics_to_compare})

df_diff = pd.merge(df_diff, df_multi_subset, on="kpi_name", how="inner")

#### Bar Plot


In [ ]:
# BAR PLOT 
# -----------------------------
# ## Ensure numeric
if type_agent == "single_agent": df = df_single_agent_qualitative
else: df = df_multi_agent_qualitative

#df = df_single_agent_qualitative
df[metric] = pd.to_numeric(df[metric], errors="coerce")
if type_agent == "single_agent": color_bar = dark_blue
else: color_bar = medium_blue

# ---- Plot ----
plt.figure(figsize=(10, 5))
plt.bar(df["kpi_name"], df[metric], color = color_bar)
#plt.xlabel(f"KPI Name", fontsize=size_axes)
plt.ylabel(f"{metric.replace('_', ' ').title()}",fontsize=size_axes)
plt.title(f" {metric.replace('_', ' ').title()} per KPI for {client_name_plot} ({type_kpi.capitalize()} )", fontsize=size_title)
plt.xticks(rotation=45, ha="right", fontsize=size_axes-3)
plt.tight_layout()
if "plot_save" in globals():
    plot_save(f"{metric}_bar_plot_{client_name_plot}_{type_kpi}_{type_agent}")
plt.show()

#### Boxplot

In [ ]:
# Boxplot PLOT (aggregate over all KPIs)
# -----------------------------

# Metrics to compare (one box each)
metrics = ["faithfulness", "context_relevance", "answer_relevance"]

# Ensure numeric + collect values across *all* KPIs
data = []
labels = []
for m in metrics:
    if m not in df.columns:
        print(f" Column '{m}' not found in df; skipping.")
        continue
    vals = pd.to_numeric(df[m], errors="coerce").dropna().values
    if len(vals) == 0:
        print(f" No numeric values for '{m}'; skipping.")
        continue
    data.append(vals)
    labels.append(m.replace("_", " ").title())

if len(data) == 0:
    print("No metric columns available to plot.")
else:
    plt.figure(figsize=(9, 5))
    bp = plt.boxplot(data, labels=labels, patch_artist=True)

    # Color the boxes (fallback to same color if fewer colors than metrics)
    colors = [dark_blue, dark_blue, dark_blue]
    if type_agent == "multi_agent":
        colors = [medium_blue, medium_blue, medium_blue]
    for i, patch in enumerate(bp["boxes"]):
        patch.set_facecolor(colors[i % len(colors)])

    plt.ylabel("Score", fontsize=size_axes)
    plt.title(f"RAGAS metric distributions (all KPIs) for {client_name_plot} ({type_kpi.capitalize()})", fontsize=size_title)
    plt.ylim(0, 1)
    plt.yticks(fontsize=size_axes)
    plt.tight_layout()
    plt.xticks(fontsize=size_axes)
    if "plot_save" in globals():
        plot_save(f"metrics_box_plot_{client_name_plot}_{type_kpi}_{type_agent}")
    plt.show()


In [ ]:
import numpy as np

# ---- Merge datasets on kpi_name to ensure alignment ----
df_multi = df_multi_agent_qualitative[["kpi_name", metric]].copy()
df_single = df_single_agent_qualitative[["kpi_name", metric]].copy()

# Rename columns to distinguish them after merge
df_single = df_single.rename(columns={metric: f"{metric}_single"})
df_multi = df_multi.rename(columns={metric: f"{metric}_multi"})

# Merge on kpi_name (inner join to only include KPIs present in both)
df_merged = pd.merge(df_single, df_multi, on="kpi_name", how="inner")

# Sort by kpi_name for consistent ordering
df_merged = df_merged.sort_values("kpi_name").reset_index(drop=True)

# ---- Check which KPIs were NOT merged ----
single_kpis = set(df_single_agent_qualitative["kpi_name"].unique())
multi_kpis = set(df_multi_agent_qualitative["kpi_name"].unique())
merged_kpis = set(df_merged["kpi_name"].unique())

only_in_single = single_kpis - multi_kpis
only_in_multi = multi_kpis - single_kpis

print(f"Merged {len(df_merged)} KPIs (single: {len(single_kpis)}, multi: {len(multi_kpis)})")

if only_in_single:
    print(f"\n  KPIs only in SINGLE agent ({len(only_in_single)}):")
    for kpi in sorted(only_in_single):
        print(f"   - {kpi}")

if only_in_multi:
    print(f"\n  KPIs only in MULTI agent ({len(only_in_multi)}):")
    for kpi in sorted(only_in_multi):
        print(f"   - {kpi}")

if not only_in_single and not only_in_multi:
    print(" All KPIs matched between both datasets!")

# ---- Plot ----
plt.figure(figsize=(12, 5))

x = np.arange(len(df_merged["kpi_name"]))
width = 0.35

plt.bar(x - width/2, df_merged[f"{metric}_single"], width, color=dark_blue, label="Single Agent")
plt.bar(x + width/2, df_merged[f"{metric}_multi"], width, color=medium_blue, label="Multi Agent")

#plt.xlabel("KPI Name", fonztsize=size_axes)
plt.ylabel(f"{metric.replace('_', ' ').capitalize()}", fontsize=size_axes)  
plt.title(f"{metric.replace('_', ' ').capitalize()} per KPI for {client_name_plot} ({type_kpi})", fontsize=size_title)
plt.xticks(x, df_merged["kpi_name"], rotation=45, ha="right", fontsize=size_axes-2)
plt.legend(title="Agent Type", loc='lower right')
plt.tight_layout()
if "plot_save" in globals():
    plot_save(f"{metric}_bar_plot_{client_name_plot}_{type_kpi}_comparison")
plt.show()

#### RADAR/SPIDER CHART 

In [ ]:
# -----------------------------
# 1. RADAR/SPIDER CHART - Compare all metrics at once
# -----------------------------
import numpy as np
import matplotlib.pyplot as plt

metrics = ["faithfulness", "context_relevance", "answer_relevance"]

# Calculate mean for each metric per agent type
single_means = [df_single_agent_qualitative[m].mean() for m in metrics]
multi_means = [df_multi_agent_qualitative[m].mean() for m in metrics]

# Radar chart setup
angles = np.linspace(0, 2 * np.pi, len(metrics), endpoint=False).tolist()
angles += angles[:1]  # Complete the circle

single_means += single_means[:1]
multi_means += multi_means[:1]

fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))

ax.plot(angles, single_means, 'o-', linewidth=2, color=dark_blue, label='Single Agent',)
ax.fill(angles, single_means, alpha=0.25, color=dark_blue)

ax.plot(angles, multi_means, 'o-', linewidth=2, color=medium_blue, label='Multi Agent')
ax.fill(angles, multi_means, alpha=0.25, color=medium_blue)

ax.set_xticks(angles[:-1])
ax.set_xticklabels([m.replace('_', ' ').title() for m in metrics], size=12)
ax.set_ylim(0, 1)
ax.set_title(f"RAGAS Metrics Comparison: {client_name_plot} ({type_kpi})", size=14, pad=20, )
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.0))

plt.tight_layout()
if "plot_save" in globals():
    plot_save(f"radar_chart_{client_name_plot}_{type_kpi}_comparison")
plt.show()

# Print summary statistics
print("\n📊 Summary Statistics:")
print("-" * 50)
print(f"{'Metric':<25} {'Single':>10} {'Multi':>10} {'Δ':>10}")
print("-" * 50)
for m in metrics:
    s = df_single_agent_qualitative[m].mean()
    mu = df_multi_agent_qualitative[m].mean()
    diff = mu - s
    arrow = "↑" if diff > 0 else "↓" if diff < 0 else "="
    print(f"{m.replace('_', ' ').title():<25} {s:>10.3f} {mu:>10.3f} {diff:>+9.3f} {arrow}")
print("-" * 50)

In [ ]:
# -----------------------------
# 2. DIFFERENCE BAR CHART - Which agent wins per KPI?
# -----------------------------




# Calculate differences (multi - single)
for m in metrics_to_compare:
    df_diff[f"{m}_diff"] = df_diff[f"{m}_multi"] - df_diff[f"{m}_single"]

# Plot difference for selected metric
fig, ax = plt.subplots(figsize=(12, 5))

diff_values = df_diff[f"{metric}_diff"]
colors = [medium_blue if v >= 0 else dark_blue for v in diff_values]

bars = ax.barh(df_diff["kpi_name"], diff_values, color=colors)
ax.axvline(x=0, color='black', linewidth=0.8, linestyle='-')

ax.set_xlabel(f"Δ {metric.replace('_', ' ').title()} (Multi - Single)", fontsize=size_axes)
ax.set_ylabel("KPI Name", fontsize=size_axes)
ax.set_title(f"Performance Difference in {metric.replace('_', ' ').title()}: {client_name_plot} ({type_kpi})", fontsize=size_subtitle)

# Add legend
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor=medium_blue, label='Multi Agent Better'),
    Patch(facecolor=dark_blue, label='Single Agent Better')
]
ax.legend(handles=legend_elements, loc='upper left', frameon=False)

plt.tight_layout()
if "plot_save" in globals():
    plot_save(f"{metric}_difference_bar_{client_name_plot}_{type_kpi}")
plt.show()

# Summary: How many KPIs does each agent win?
multi_wins = (diff_values > 0).sum()
single_wins = (diff_values < 0).sum()
ties = (diff_values == 0).sum()
print(f"\n🏆 Winner Summary for {metric.replace('_', ' ').title()}:")
print(f"   Multi-Agent wins: {multi_wins}/{len(diff_values)} KPIs")
print(f"   Single-Agent wins: {single_wins}/{len(diff_values)} KPIs")
print(f"   Ties: {ties}/{len(diff_values)} KPIs")

#### SCATTER PLOT

In [ ]:
# -----------------------------
# 3. SCATTER PLOT - Single vs Multi (Diagonal = Equal Performance)
# -----------------------------

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, m in zip(axes, metrics_to_compare):
    x = df_diff[f"{m}_single"]
    y = df_diff[f"{m}_multi"]
    
    ax.scatter(x, y, s=80, alpha=1, color=dark_blue, edgecolor='white', linewidth=0.5)
    
    # Diagonal line (y=x means equal performance)
    ax.plot([0, 1], [0, 1], 'k--', alpha=1.0, label='Equal Performance', color=dark_blue)
    
    # Fill regions
    ax.fill_between([0, 1], [0, 1], [1, 1], alpha=0.6, color=light_blue, label='Multi Better')
    ax.fill_between([0, 1], [0, 0], [0, 1], alpha=0.6, color=medium_blue, label='Single Better')
    
    ax.set_xlabel(f"Single Agent {m.replace('_', ' ').title()}", fontsize=size_axes)
    ax.set_ylabel(f"Multi Agent {m.replace('_', ' ').title()}", fontsize=size_axes)
    ax.set_title(m.replace('_', ' ').title(), fontsize=size_subtitle, fontweight = "bold")
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.set_aspect('equal')
    
    # Add correlation
    corr = np.corrcoef(x, y)[0, 1]
    ax.text(0.05, 0.95, f"r = {corr:.2f}", transform=ax.transAxes, fontsize=size_axes   ,
            verticalalignment='top', bbox=dict(boxstyle='round', facecolor='white', alpha=1))

plt.suptitle(f"Single vs Multi Agent Performance: {client_name_plot} ({type_kpi})", fontsize=size_title, y=1.02)
plt.tight_layout()
if "plot_save" in globals():
    plot_save(f"scatter_comparison_{client_name_plot}_{type_kpi}")
plt.show()

#### BOX PLOT - Distribution Comparison

In [ ]:
# -----------------------------
# 4. BOX PLOT - Distribution Comparison
# -----------------------------

fig, axes = plt.subplots(1, 3, figsize=(14, 5))

for ax, m in zip(axes, metrics_to_compare):
    data_single = df_single_agent_qualitative[m].dropna()
    data_multi = df_multi_agent_qualitative[m].dropna()
    
    bp = ax.boxplot(
        [data_single, data_multi],
        labels=['Single Agent', 'Multi Agent'],
        patch_artist=True,
        widths=0.6,
        medianprops={"color": "orange", "linewidth": 2}
    )
    
    # Color the boxes
    bp['boxes'][0].set_facecolor(dark_blue)
    bp['boxes'][1].set_facecolor(medium_blue)
    for box in bp['boxes']:
        box.set_alpha(0.9)
    
    ax.set_ylabel(m.replace('_', ' ').title(), fontsize=size_axes+1)
    ax.set_ylim(0, 1.05)
    ax.set_title(m.replace('_', ' ').title(),fontweight='bold', fontsize=size_subtitle)
    ax.tick_params(axis='x', labelsize=size_axes)
    ax.tick_params(axis='y', labelsize=size_axes)
    
    # Add mean markers
    ax.scatter([1, 2], [data_single.mean(), data_multi.mean()], 
               marker='D', color='white', s=50, zorder=3, edgecolor='black')

plt.suptitle(f"Score Distribution: {client_name_plot} ({type_kpi})", fontsize=size_title, y=1.02)
plt.tight_layout()
if "plot_save" in globals():
    plot_save(f"boxplot_comparison_{client_name_plot}_{type_kpi}")
plt.show()

#### HEATMAP - Side-by-Side Performance Matrix

In [ ]:
# -----------------------------
# 5. HEATMAP - Side-by-Side Performance Matrix
# -----------------------------

# Prepare data for heatmap (uses df_diff from cell 2 which has all metrics)
heatmap_single = df_diff.set_index('kpi_name')[[f"{m}_single" for m in metrics_to_compare]]
heatmap_multi = df_diff.set_index('kpi_name')[[f"{m}_multi" for m in metrics_to_compare]]

# Rename columns for cleaner display
heatmap_single.columns = [m.replace('_', ' ').title() for m in metrics_to_compare]
heatmap_multi.columns = [m.replace('_', ' ').title() for m in metrics_to_compare]

fig, axes = plt.subplots(1, 2, figsize=(14, max(6, len(df_diff) * 0.4)))

# Single Agent Heatmap
im1 = axes[0].imshow(heatmap_single.values, cmap='Blues', aspect='auto', vmin=0, vmax=1)
axes[0].set_xticks(range(len(heatmap_single.columns)))
axes[0].set_xticklabels(heatmap_single.columns, rotation=45, ha='right', fontsize=size_axes)
axes[0].set_yticks(range(len(heatmap_single.index)))
axes[0].set_yticklabels(heatmap_single.index, fontsize=size_axes)
axes[0].set_title('Single Agent', fontsize=size_subtitle, fontweight='bold')

# Add value annotations
for i in range(len(heatmap_single.index)):
    for j in range(len(heatmap_single.columns)):
        val = heatmap_single.values[i, j]
        color = 'white' if val > 0.5 else 'black'
        axes[0].text(j, i, f'{val:.2f}', ha='center', va='center', color=color, fontsize=size_axes-2)

# Multi Agent Heatmap
im2 = axes[1].imshow(heatmap_multi.values, cmap='Blues', aspect='auto', vmin=0, vmax=1)
axes[1].set_xticks(range(len(heatmap_multi.columns)))
axes[1].set_xticklabels(heatmap_multi.columns, rotation=45, ha='right', fontsize=size_axes)
axes[1].set_yticks(range(len(heatmap_multi.index)))
axes[1].set_yticklabels([])  # Remove y-axis labels from multi agent
axes[1].set_title('Multi Agent', fontsize=size_subtitle, fontweight='bold')

# Add value annotations
for i in range(len(heatmap_multi.index)):
    for j in range(len(heatmap_multi.columns)):
        val = heatmap_multi.values[i, j]
        color = 'white' if val > 0.5 else 'black'
        axes[1].text(j, i, f'{val:.2f}', ha='center', va='center', color=color, fontsize=size_axes-2)

plt.suptitle(f"Performance Heatmap: {client_name_plot} ({type_kpi})", fontsize=size_title, y=1.02)

# Add vertical colorbar next to multi agent plot
fig.colorbar(im2, ax=axes[1], orientation='vertical', shrink=0.8, label='Score', pad=0.02)

plt.tight_layout()
if "plot_save" in globals():
    plot_save(f"heatmap_comparison_{client_name_plot}_{type_kpi}")
plt.show()


####  LOLLIPOP CHART - Difference by KPI 

In [ ]:
# -----------------------------
# 6. LOLLIPOP CHART - Difference by KPI 
# -----------------------------

fig, axes = plt.subplots(1, 3, figsize=(16, max(5, len(df_diff) * 0.35)))

for ax, m in zip(axes, metrics_to_compare):
    diff = df_diff[f'{m}_multi'] - df_diff[f'{m}_single']
    kpi_names = df_diff['kpi_name'].values
    
    # Sort by difference
    sorted_idx = diff.argsort()
    diff_sorted = diff.iloc[sorted_idx]
    kpi_sorted = kpi_names[sorted_idx]
    
    colors = [medium_blue if d > 0 else dark_blue for d in diff_sorted]
    
    # Draw lollipop stems
    for i, (kpi, d, c) in enumerate(zip(kpi_sorted, diff_sorted, colors)):
        ax.hlines(y=i, xmin=0, xmax=d, color=c, linewidth=2, alpha=0.7)
        ax.scatter(d, i, color=c, s=80, zorder=3)
    
    ax.axvline(x=0, color='black', linewidth=0.8, linestyle='-')
    ax.set_yticks(range(len(kpi_sorted)))
    ax.tick_params(axis='x', labelsize=size_axes)
    ax.set_yticklabels(kpi_sorted, fontsize=size_axes)
    ax.set_xlabel('Difference (Multi - Single)', fontsize=size_axes)
    ax.set_title(m.replace('_', ' ').title(), fontsize=size_subtitle+1, fontweight='bold')
    ax.set_xlim(-1, 1, auto=True)

# Add legend
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor=medium_blue, label='Multi Agent Better'),
    Patch(facecolor=dark_blue, label='Single Agent Better')
]
fig.legend(handles=legend_elements, loc='upper center', ncol=2, bbox_to_anchor=(0.5, 1.08), fontsize=size_axes+1, frameon=False)

plt.suptitle(f"Performance Difference: {client_name_plot} ({type_kpi})", fontsize=size_title+1, y=1.12)
plt.tight_layout()
if "plot_save" in globals():
    plot_save(f"lollipop_comparison_{client_name_plot}_{type_kpi}")
plt.show()